# Milestone 2: Data Quality dan Profiling

Notebook ini melanjutkan profiling Indonesian News AZ dari Milestone 1. Alur mempertahankan tujuh bagian utama template dosen: profiling DuckDB, deteksi missing values, penanganan missing values, outlier, deduplikasi, validasi, dan penyimpanan hasil. Data mentah tidak ditimpa. Semua pemindaian memakai seluruh file Parquet lokal secara rekursif dengan Polars lazy evaluation.

Sumber dataset: [esteler-ai/idn-news-az di Hugging Face](https://huggingface.co/datasets/esteler-ai/idn-news-az).


In [1]:
from pathlib import Path
import hashlib
import json

import duckdb
import polars as pl

EXPECTED_FILES = 441
EXPECTED_ROWS = 1_149_789
EXPECTED_COLUMNS = ["date", "link", "title", "text"]

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / "data" / "raw").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder data/raw tidak ditemukan dari direktori kerja saat ini atau induknya.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "indonesian_news_az_clean.parquet"
parquet_paths = sorted(path for path in RAW_DIR.rglob("*.parquet") if path.is_file())
if not parquet_paths:
    raise FileNotFoundError(f"Tidak ada file Parquet di {RAW_DIR}")

# Hash streaming dipakai untuk memastikan file sumber tidak berubah selama notebook berjalan.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

raw_manifest_before = {
    path.relative_to(RAW_DIR).as_posix(): (path.stat().st_size, sha256_file(path))
    for path in parquet_paths
}
raw_size_bytes = sum(size for size, _ in raw_manifest_before.values())
news = pl.scan_parquet([str(path) for path in parquet_paths], low_memory=True)
raw_schema = news.collect_schema()
raw_rows = news.select(pl.len().alias("jumlah_baris")).collect(engine="streaming").item()

assert len(parquet_paths) == EXPECTED_FILES, f"Jumlah file tidak sesuai: {len(parquet_paths)}"
assert raw_schema.names() == EXPECTED_COLUMNS, f"Kolom tidak sesuai: {raw_schema.names()}"
assert raw_rows == EXPECTED_ROWS, f"Jumlah baris tidak sesuai: {raw_rows}"

print(f"Root proyek: {PROJECT_ROOT}")
print(f"File Parquet: {len(parquet_paths):,}; baris: {raw_rows:,}; kolom: {len(raw_schema)}")
print(f"Ukuran mentah: {raw_size_bytes:,} byte")
print("Tipe data:", dict(raw_schema))
print(f"Manifest SHA-256 sebelum cleaning dibuat untuk {len(raw_manifest_before):,} file.")


Root proyek: C:\Users\WINDOWS 11\OneDrive\tugas1-abd-2026
File Parquet: 441; baris: 1,149,789; kolom: 4
Ukuran mentah: 1,558,347,901 byte
Tipe data: {'date': String, 'link': String, 'title': String, 'text': String}
Manifest SHA-256 sebelum cleaning dibuat untuk 441 file.


## 1. Profiling dengan DuckDB `SUMMARIZE`

DuckDB membaca langsung 441 file Parquet dan menjalankan `SUMMARIZE` atas seluruh baris. Tidak ada konversi seluruh dataset ke Pandas. Kolom `min`/`max` untuk teks merupakan nilai leksikografis, bukan ukuran panjang atau ringkasan semantik; ringkasan teks yang relevan dihitung dengan panjang karakter pada bagian outlier.


In [2]:
connection = duckdb.connect()
raw_relation = connection.read_parquet([str(path) for path in parquet_paths])
raw_relation.create_view("indonesian_news_raw")

summary_cursor = connection.execute("SUMMARIZE SELECT * FROM indonesian_news_raw")
summary_columns = [item[0] for item in summary_cursor.description]
summary_rows = summary_cursor.fetchall()
summary_raw = pl.DataFrame(summary_rows, schema=summary_columns, orient="row", infer_schema_length=None)
# Potong preview min/max agar isi artikel panjang tidak membanjiri output notebook.
summarize_display = summary_raw.with_columns([
    pl.col(column).cast(pl.String).str.replace_all(r"\s+", " ").str.slice(0, 100).alias(column)
    for column in ("min", "max")
])
summarize_display.select([
    "column_name", "column_type", "count", "null_percentage", "approx_unique", "min", "max",
]).to_dicts()


[{'column_name': 'date',
  'column_type': 'VARCHAR',
  'count': 1149789,
  'null_percentage': Decimal('0.00'),
  'approx_unique': 448,
  'min': '2019-05-10',
  'max': 'NaT'},
 {'column_name': 'link',
  'column_type': 'VARCHAR',
  'count': 1149789,
  'null_percentage': Decimal('0.00'),
  'approx_unique': 1019058,
  'min': 'https://20.detik.com/blak-blakan/20230620-230620041/sodetan-ciliwung-kado-ulang-tahun-jakarta-ke-496',
  'max': 'https://yogyakarta.kompas.com/read/2023/10/01/151321878/jadwal-krl-solo-jogja-oktober-2023-lengkap-d'},
 {'column_name': 'title',
  'column_type': 'VARCHAR',
  'count': 1149789,
  'null_percentage': Decimal('0.00'),
  'approx_unique': 1031875,
  'min': '"100% Manusia Film Festival 2023" siap putar 62 film dari 18 negara',
  'max': '\ufeffUnsur-unsur Surat Pribadi, Lengkap dengan Ciri Kebahasaannya'},
 {'column_name': 'text',
  'column_type': 'VARCHAR',
  'count': 1149789,
  'null_percentage': Decimal('0.00'),
  'approx_unique': 971628,
  'min': ' Okezone Up

## 2. Deteksi Missing Values

Null asli, string kosong/whitespace, dan nilai tanggal yang gagal diparsing dihitung terpisah pada seluruh dataset. Contoh tanggal invalid ditampilkan sebelum menetapkan aturan. Nilai `NaT` bukan null Parquet: ia adalah string yang gagal dikonversi ke format tanggal.


In [3]:
raw_quality = news.select([
    *[pl.col(column).is_null().sum().alias(f"null_{column}") for column in EXPECTED_COLUMNS],
    *[(pl.col(column).str.strip_chars() == "").fill_null(False).sum().alias(f"kosong_{column}")
      for column in EXPECTED_COLUMNS],
]).collect(engine="streaming")

parsed_date = pl.col("date").str.strptime(pl.Date, "%Y-%m-%d", strict=False)
date_quality = news.select([
    pl.col("date").is_null().sum().alias("tanggal_null_asli"),
    (pl.col("date").is_not_null() & parsed_date.is_null()).sum().alias("tanggal_gagal_diparsing"),
    parsed_date.is_not_null().sum().alias("tanggal_valid"),
    parsed_date.min().alias("tanggal_valid_min"),
    parsed_date.max().alias("tanggal_valid_max"),
]).collect(engine="streaming")
invalid_date_examples = (
    news.filter(pl.col("date").is_not_null() & parsed_date.is_null())
    .select("date").unique().head(10).collect(engine="streaming")
)

print("Null dan string kosong/whitespace:")
print(raw_quality)
print("Ringkasan tanggal:")
print(date_quality)
print("Contoh nilai date yang gagal diparsing:", invalid_date_examples.get_column("date").to_list())


Null dan string kosong/whitespace:
shape: (1, 8)
┌───────────┬───────────┬────────────┬───────────┬────────────┬────────────┬───────────┬───────────┐
│ null_date ┆ null_link ┆ null_title ┆ null_text ┆ kosong_dat ┆ kosong_lin ┆ kosong_ti ┆ kosong_te │
│ ---       ┆ ---       ┆ ---        ┆ ---       ┆ e          ┆ k          ┆ tle       ┆ xt        │
│ u32       ┆ u32       ┆ u32        ┆ u32       ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│           ┆           ┆            ┆           ┆ u32        ┆ u32        ┆ u32       ┆ u32       │
╞═══════════╪═══════════╪════════════╪═══════════╪════════════╪════════════╪═══════════╪═══════════╡
│ 0         ┆ 0         ┆ 0          ┆ 0         ┆ 0          ┆ 0          ┆ 0         ┆ 0         │
└───────────┴───────────┴────────────┴───────────┴────────────┴────────────┴───────────┴───────────┘
Ringkasan tanggal:
shape: (1, 5)
┌───────────────────┬──────────────────────┬───────────────┬───────────────────┬───────────────────┐
│ tanggal

## 3. Penanganan Missing Values

Profiling lokal menjadi dasar keputusan: bila judul, isi, atau URL null/kosong, baris tersebut tidak menyediakan artikel yang dapat digunakan dan akan dikecualikan tanpa mengisi nilai buatan. URL harus memiliki skema HTTP(S) dan host. Tanggal invalid tidak diisi dan tidak menyebabkan penghapusan: kolom asli `date` dipertahankan, sementara `date_parsed` dan `date_status` ditambahkan untuk audit.


In [4]:
url_is_usable = pl.col("link").is_not_null() & pl.col("link").str.contains(r"(?i)^https?://[^/?#]+").fill_null(False)
title_is_usable = pl.col("title").is_not_null() & (pl.col("title").str.strip_chars() != "").fill_null(False)
text_is_usable = pl.col("text").is_not_null() & (pl.col("text").str.strip_chars() != "").fill_null(False)

df_missing_handled = news.with_columns([
    parsed_date.alias("date_parsed"),
    pl.when(pl.col("date").is_null()).then(pl.lit("null_asli"))
      .when(parsed_date.is_null()).then(pl.lit("gagal_diparsing"))
      .otherwise(pl.lit("valid")).alias("date_status"),
    pl.col("title").str.len_chars().alias("title_length"),
    pl.col("text").str.len_chars().alias("text_length"),
])

quality_issues = df_missing_handled.select([
    (~title_is_usable).sum().alias("judul_tidak_dapat_digunakan"),
    (~text_is_usable).sum().alias("isi_tidak_dapat_digunakan"),
    (~url_is_usable).sum().alias("url_tidak_dapat_digunakan"),
]).collect(engine="streaming")
df_eligible = df_missing_handled.filter(title_is_usable & text_is_usable & url_is_usable)
eligible_rows = df_eligible.select(pl.len()).collect(engine="streaming").item()

print("Baris yang dikecualikan karena judul/isi/URL tidak dapat digunakan:")
print(quality_issues)
print(f"Baris setelah aturan kelayakan: {eligible_rows:,} dari {raw_rows:,}")
print("date asli tidak diubah; tanggal gagal diparsing disimpan dengan status eksplisit.")


Baris yang dikecualikan karena judul/isi/URL tidak dapat digunakan:
shape: (1, 3)
┌─────────────────────────────┬───────────────────────────┬───────────────────────────┐
│ judul_tidak_dapat_digunakan ┆ isi_tidak_dapat_digunakan ┆ url_tidak_dapat_digunakan │
│ ---                         ┆ ---                       ┆ ---                       │
│ u32                         ┆ u32                       ┆ u32                       │
╞═════════════════════════════╪═══════════════════════════╪═══════════════════════════╡
│ 0                           ┆ 0                         ┆ 0                         │
└─────────────────────────────┴───────────────────────────┴───────────────────────────┘
Baris setelah aturan kelayakan: 1,149,789 dari 1,149,789
date asli tidak diubah; tanggal gagal diparsing disimpan dengan status eksplisit.


## 4. Deteksi dan Penanganan Outlier

Dataset tidak mempunyai kolom numerik asli. `title_length` dan `text_length` dihitung sebagai fitur turunan, lalu Q1, Q3, IQR, pagar 1,5?IQR, jumlah, dan persentase kandidat dihitung dari seluruh baris. Tidak dilakukan winsorization atau penghapusan: panjang artikel yang ekstrem dapat merupakan variasi editorial yang sah. Kandidat ditandai untuk peninjauan lanjutan.


In [5]:
quartile_summary = df_eligible.select([
    pl.col("title_length").quantile(0.25, interpolation="linear").alias("title_q1"),
    pl.col("title_length").quantile(0.75, interpolation="linear").alias("title_q3"),
    pl.col("text_length").quantile(0.25, interpolation="linear").alias("text_q1"),
    pl.col("text_length").quantile(0.75, interpolation="linear").alias("text_q3"),
]).collect(engine="streaming").row(0, named=True)

iqr_bounds = {}
for column in ("title_length", "text_length"):
    q1 = quartile_summary[f"{column.replace('_length', '')}_q1"]
    q3 = quartile_summary[f"{column.replace('_length', '')}_q3"]
    iqr = q3 - q1
    iqr_bounds[column] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

outlier_expressions = []
for column, (lower, upper) in iqr_bounds.items():
    outlier_expressions.extend([
        ((pl.col(column) < lower) | (pl.col(column) > upper)).sum().alias(f"{column}_kandidat"),
        (((pl.col(column) < lower) | (pl.col(column) > upper)).sum() / pl.len() * 100).alias(f"{column}_persen"),
    ])
outlier_counts = df_eligible.select(outlier_expressions).collect(engine="streaming")
outlier_report = pl.DataFrame([
    {"fitur": column,
     "Q1": quartile_summary[f"{column.replace('_length', '')}_q1"],
     "Q3": quartile_summary[f"{column.replace('_length', '')}_q3"],
     "IQR": quartile_summary[f"{column.replace('_length', '')}_q3"] - quartile_summary[f"{column.replace('_length', '')}_q1"],
     "batas_bawah": iqr_bounds[column][0], "batas_atas": iqr_bounds[column][1],
     "jumlah_kandidat": outlier_counts.item(0, f"{column}_kandidat"),
     "persen_kandidat": outlier_counts.item(0, f"{column}_persen")}
    for column in ("title_length", "text_length")
])

# Tidak ada nilai panjang yang dipotong atau dibuang pada tahap ini.
df_outlier_handled = df_eligible
outlier_report


fitur,Q1,Q3,IQR,batas_bawah,batas_atas,jumlah_kandidat,persen_kandidat
str,f64,f64,f64,f64,f64,i64,f64
"""title_length""",59.0,81.0,22.0,26.0,114.0,13872,1.206482
"""text_length""",1492.0,2942.0,1450.0,-683.0,5117.0,47293,4.113189


## 5. Deduplikasi

`link` digunakan sebagai kunci identitas kandidat. Jumlah baris berlebih dari duplikasi identik (semua empat kolom asli) dibandingkan dengan baris berlebih akibat URL berulang. Grup URL diperiksa untuk konflik judul, isi, dan tanggal.

**Aturan deterministik:** pertahankan kemunculan pertama untuk setiap URL, dengan daftar file diproses dalam urutan path leksikografis dan urutan baris sumber dipertahankan (`keep="first", maintain_order=True`). Aturan ini tidak menganggap semua versi URL sama isi. Untuk 86 grup dengan konflik judul/isi, pilihan pertama dipertahankan secara konsisten; pilihan tersebut tidak selalu versi terbaru atau terlengkap, dan ditandai sebagai keterbatasan.


In [6]:
exact_duplicate_groups = (
    news.group_by(pl.all()).len()
    .filter(pl.col("len") > 1)
    .select([
        pl.len().alias("grup_duplikasi_identik"),
        (pl.col("len") - 1).sum().fill_null(0).alias("baris_identik_berlebih"),
    ])
    .collect(engine="streaming")
)
url_groups = (
    df_eligible.group_by("link").agg([
        pl.len().alias("jumlah_baris"),
        pl.col("title").n_unique().alias("jumlah_judul"),
        pl.col("text").n_unique().alias("jumlah_isi"),
        pl.col("date").n_unique().alias("jumlah_tanggal"),
    ])
    .filter(pl.col("jumlah_baris") > 1)
)
url_duplicate_summary = url_groups.select([
    pl.len().alias("grup_url_berulang"),
    (pl.col("jumlah_baris") - 1).sum().alias("baris_url_berlebih"),
    ((pl.col("jumlah_judul") > 1) | (pl.col("jumlah_isi") > 1)).sum().alias("grup_konflik_konten"),
    (pl.col("jumlah_tanggal") > 1).sum().alias("grup_konflik_tanggal"),
]).collect(engine="streaming")
content_conflict_examples = (
    url_groups.filter((pl.col("jumlah_judul") > 1) | (pl.col("jumlah_isi") > 1))
    .select(["link", "jumlah_baris", "jumlah_judul", "jumlah_isi", "jumlah_tanggal"])
    .head(5).collect(engine="streaming")
)

print("Duplikasi identik seluruh kolom:")
print(exact_duplicate_groups)
print("Duplikasi URL dan konflik:")
print(url_duplicate_summary)
print("Contoh grup URL dengan konflik konten (maksimum lima):")
content_conflict_examples


Duplikasi identik seluruh kolom:
shape: (1, 2)
┌────────────────────────┬────────────────────────┐
│ grup_duplikasi_identik ┆ baris_identik_berlebih │
│ ---                    ┆ ---                    │
│ u32                    ┆ u32                    │
╞════════════════════════╪════════════════════════╡
│ 11457                  ┆ 108391                 │
└────────────────────────┴────────────────────────┘
Duplikasi URL dan konflik:
shape: (1, 4)
┌───────────────────┬────────────────────┬─────────────────────┬──────────────────────┐
│ grup_url_berulang ┆ baris_url_berlebih ┆ grup_konflik_konten ┆ grup_konflik_tanggal │
│ ---               ┆ ---                ┆ ---                 ┆ ---                  │
│ u32               ┆ u32                ┆ u32                 ┆ u32                  │
╞═══════════════════╪════════════════════╪═════════════════════╪══════════════════════╡
│ 9017              ┆ 120304             ┆ 86                  ┆ 157                  │
└───────────────────

link,jumlah_baris,jumlah_judul,jumlah_isi,jumlah_tanggal
str,u32,u32,u32,u32
"""https://www.metrotvnews.com/in…",2,1,2,1
"""https://www.metrotvnews.com/in…",3,1,2,1
"""https://otomotif.tempo.co/read…",2,1,2,1
"""https://www.metrotvnews.com/in…",2,1,2,1
"""https://www.cnbcindonesia.com/…",2,1,2,1


In [7]:
df_dedup = df_outlier_handled.unique(
    subset=["link"], keep="first", maintain_order=True
)
rows_before_dedup = eligible_rows
rows_after_dedup = df_dedup.select(pl.len()).collect(engine="streaming").item()
removed_by_dedup = rows_before_dedup - rows_after_dedup

print(f"Baris sebelum deduplikasi URL: {rows_before_dedup:,}")
print(f"Baris sesudah deduplikasi URL: {rows_after_dedup:,}")
print(f"Baris yang dihapus oleh aturan URL: {removed_by_dedup:,}")
print(f"Duplikasi URL setelah kebijakan: {rows_after_dedup - df_dedup.select(pl.col('link').n_unique()).collect(engine='streaming').item():,}")


Baris sebelum deduplikasi URL: 1,149,789
Baris sesudah deduplikasi URL: 1,029,485
Baris yang dihapus oleh aturan URL: 120,304


Duplikasi URL setelah kebijakan: 0


## 6. Validasi Data Dasar

Validasi memastikan kolom wajib tersedia, judul/isi/URL dapat digunakan, URL berformat HTTP(S), dan URL unik setelah deduplikasi. `date_parsed` yang null karena `date_status="gagal_diparsing"` merupakan pengecualian terdokumentasi yang dipertahankan, bukan pelanggaran yang disembunyikan. Kandidat IQR juga dipertahankan dan dilaporkan.


In [8]:
required_clean_columns = set(EXPECTED_COLUMNS + ["date_parsed", "date_status", "title_length", "text_length"])
clean_schema = df_dedup.collect_schema()
missing_required_columns = sorted(required_clean_columns - set(clean_schema.names()))
assert not missing_required_columns, f"Kolom wajib hilang: {missing_required_columns}"
assert removed_by_dedup == url_duplicate_summary.item(0, "baris_url_berlebih"), "Penghapusan URL tidak sesuai dengan ringkasan grup berulang."
assert raw_rows - rows_after_dedup == (raw_rows - eligible_rows) + removed_by_dedup, "Perubahan jumlah baris tidak sesuai alasan yang dicatat."

print("Kolom wajib tersedia:", sorted(required_clean_columns))
print(f"Baris invalid yang dikecualikan karena judul/isi/URL: {raw_rows - eligible_rows:,}")
invalid_date_after_dedup = df_dedup.filter(pl.col("date_status") == "gagal_diparsing").select(pl.len()).collect(engine="streaming").item()
print(f"Tanggal gagal parsing awal: {date_quality.item(0, 'tanggal_gagal_diparsing'):,}; setelah deduplikasi: {invalid_date_after_dedup:,}.")
print("Aturan validasi URL sesudah deduplikasi akan diperiksa dengan DuckDB setelah file disimpan.")


Kolom wajib tersedia: ['date', 'date_parsed', 'date_status', 'link', 'text', 'text_length', 'title', 'title_length']
Baris invalid yang dikecualikan karena judul/isi/URL: 0


Tanggal gagal parsing awal: 101,421; setelah deduplikasi: 101,403.
Aturan validasi URL sesudah deduplikasi akan diperiksa dengan DuckDB setelah file disimpan.


## 7. Simpan Data Bersih

Hasil disimpan ke `data/processed/indonesian_news_az_clean.parquet` melalui `sink_parquet()` agar hasil akhir tidak dikumpulkan seluruhnya ke RAM. Sesudahnya file dibuka kembali untuk memeriksa jumlah baris dan schema. Ringkasan validasi final dijalankan dengan SQL DuckDB terhadap berkas hasil.


In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
required_output_names = EXPECTED_COLUMNS + ["date_parsed", "date_status", "title_length", "text_length"]
reuse_existing_output = False
if OUTPUT_PATH.exists():
    try:
        existing_news = pl.scan_parquet(str(OUTPUT_PATH), low_memory=True)
        existing_schema = existing_news.collect_schema()
        existing_rows = existing_news.select(pl.len()).collect(engine="streaming").item()
        reuse_existing_output = existing_rows == rows_after_dedup and existing_schema.names() == required_output_names
    except Exception as error:
        print(f"Output sebelumnya tidak dapat dibaca dan akan dibuat ulang: {error}")
    if not reuse_existing_output:
        OUTPUT_PATH.unlink()

if reuse_existing_output:
    print("Output Parquet yang ada memiliki schema dan jumlah baris yang diharapkan; tidak ditulis ulang.")
else:
    df_dedup.sink_parquet(OUTPUT_PATH, compression="zstd", engine="streaming")

clean_news = pl.scan_parquet(str(OUTPUT_PATH), low_memory=True)
clean_schema = clean_news.collect_schema()
clean_rows = clean_news.select(pl.len().alias("jumlah_baris")).collect(engine="streaming").item()
assert clean_rows == rows_after_dedup, f"Jumlah baris baca-ulang {clean_rows:,} tidak sama dengan {rows_after_dedup:,}"
assert set(EXPECTED_COLUMNS).issubset(clean_schema.names()), "Kolom asli tidak lengkap pada file hasil."

clean_relation = connection.read_parquet(str(OUTPUT_PATH))
clean_relation.create_view("indonesian_news_clean")
validation_sql = """
SELECT
    COUNT(*) AS total_baris,
    SUM(CASE WHEN title IS NULL OR trim(title) = '' THEN 1 ELSE 0 END) AS judul_null_atau_kosong,
    SUM(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) AS isi_null_atau_kosong,
    SUM(CASE WHEN link IS NULL OR trim(link) = '' OR NOT regexp_matches(link, '(?i)^https?://[^/?#]+') THEN 1 ELSE 0 END) AS url_tidak_layak,
    (SELECT COUNT(*) FROM (SELECT link FROM indonesian_news_clean GROUP BY link HAVING COUNT(*) > 1)) AS grup_url_berulang,
    SUM(CASE WHEN date_status = 'gagal_diparsing' THEN 1 ELSE 0 END) AS tanggal_invalid_dipertahankan,
    SUM(CASE WHEN date_status = 'null_asli' THEN 1 ELSE 0 END) AS tanggal_null_asli,
    SUM(CASE WHEN title_length < ? OR title_length > ? THEN 1 ELSE 0 END) AS kandidat_iqr_judul_dipertahankan,
    SUM(CASE WHEN text_length < ? OR text_length > ? THEN 1 ELSE 0 END) AS kandidat_iqr_isi_dipertahankan
FROM indonesian_news_clean
"""
query_parameters = [iqr_bounds["title_length"][0], iqr_bounds["title_length"][1],
                    iqr_bounds["text_length"][0], iqr_bounds["text_length"][1]]
validation_cursor = connection.execute(validation_sql, query_parameters)
validation_columns = [item[0] for item in validation_cursor.description]
validation_result = pl.DataFrame(validation_cursor.fetchall(), schema=validation_columns, orient="row")
assert validation_result.item(0, "total_baris") == clean_rows
assert validation_result.item(0, "judul_null_atau_kosong") == 0
assert validation_result.item(0, "isi_null_atau_kosong") == 0
assert validation_result.item(0, "url_tidak_layak") == 0
assert validation_result.item(0, "grup_url_berulang") == 0
assert validation_result.item(0, "tanggal_invalid_dipertahankan") == invalid_date_after_dedup

print(f"Berkas hasil: {OUTPUT_PATH}")
print(f"Ukuran hasil: {OUTPUT_PATH.stat().st_size:,} byte")
print(f"Jumlah baris baca-ulang: {clean_rows:,}")
print("Schema hasil:", dict(clean_schema))
print("Validasi akhir DuckDB:")
print(validation_result.row(0, named=True))
validation_result


Output Parquet yang ada memiliki schema dan jumlah baris yang diharapkan; tidak ditulis ulang.


Berkas hasil: C:\Users\WINDOWS 11\OneDrive\tugas1-abd-2026\data\processed\indonesian_news_az_clean.parquet
Ukuran hasil: 803,849,540 byte
Jumlah baris baca-ulang: 1,029,485
Schema hasil: {'date': String, 'link': String, 'title': String, 'text': String, 'date_parsed': Date, 'date_status': String, 'title_length': UInt32, 'text_length': UInt32}
Validasi akhir DuckDB:
{'total_baris': 1029485, 'judul_null_atau_kosong': 0, 'isi_null_atau_kosong': 0, 'url_tidak_layak': 0, 'grup_url_berulang': 0, 'tanggal_invalid_dipertahankan': 101403, 'tanggal_null_asli': 0, 'kandidat_iqr_judul_dipertahankan': 13709, 'kandidat_iqr_isi_dipertahankan': 37559}


total_baris,judul_null_atau_kosong,isi_null_atau_kosong,url_tidak_layak,grup_url_berulang,tanggal_invalid_dipertahankan,tanggal_null_asli,kandidat_iqr_judul_dipertahankan,kandidat_iqr_isi_dipertahankan
i64,i64,i64,i64,i64,i64,i64,i64,i64
1029485,0,0,0,0,101403,0,13709,37559


In [10]:
# Bandingkan seluruh nama, ukuran, dan SHA-256 file sumber setelah cleaning.
raw_manifest_after = {
    path.relative_to(RAW_DIR).as_posix(): (path.stat().st_size, sha256_file(path))
    for path in sorted(path for path in RAW_DIR.rglob("*.parquet") if path.is_file())
}
raw_unchanged = raw_manifest_before == raw_manifest_after
assert raw_unchanged, "Integritas data mentah berubah selama notebook berjalan."
print(f"Jumlah file mentah setelah cleaning: {len(raw_manifest_after):,}")
print(f"Checksum SHA-256 semua file mentah sama sebelum/sesudah: {raw_unchanged}")
print(f"Total baris awal: {raw_rows:,}; akhir: {clean_rows:,}; berkurang: {raw_rows - clean_rows:,}.")


Jumlah file mentah setelah cleaning: 441
Checksum SHA-256 semua file mentah sama sebelum/sesudah: True
Total baris awal: 1,149,789; akhir: 1,029,485; berkurang: 120,304.


## Catatan Milestone 2

Catatan berikut diisi menggunakan hasil query notebook: jumlah awal/akhir, missing values, tanggal invalid, kandidat IQR, duplikasi identik/URL, konflik URL, dan hasil validasi. Tidak ada imputasi tanggal atau winsorization. Baris berkurang hanya karena aturan judul/isi/URL yang tidak dapat digunakan (bila ditemukan) dan deduplikasi URL.

Keterbatasan keputusan: URL berulang yang memiliki isi berbeda diputuskan dengan mempertahankan kemunculan pertama pada urutan file leksikografis dan urutan baris sumber. Ini membuat proses dapat diulang, tetapi tidak menjamin versi terlengkap/terbaru. Tanggal `NaT` tetap tersedia dalam `date` asli dan ditandai pada `date_status`; nilai ini memerlukan peninjauan sebelum analisis temporal. Kandidat panjang berdasarkan IQR tidak otomatis salah, sehingga dipertahankan tanpa pemotongan.


In [11]:
print(f"Baris awal: {raw_rows:,}; baris akhir: {clean_rows:,}.")
print(f"Baris dikecualikan karena kelayakan: {raw_rows - eligible_rows:,}; baris dihapus karena URL berulang: {removed_by_dedup:,}.")
print(f"Null asli per kolom: {raw_quality.row(0, named=True)}")
print(f"Tanggal gagal parsing: {date_quality.item(0, 'tanggal_gagal_diparsing'):,} sebelum; {validation_result.item(0, 'tanggal_invalid_dipertahankan'):,} setelah deduplikasi URL; contoh: {invalid_date_examples.get_column('date').to_list()}.")
print(f"Duplikasi identik berlebih: {exact_duplicate_groups.item(0, 'baris_identik_berlebih'):,}; URL berlebih: {url_duplicate_summary.item(0, 'baris_url_berlebih'):,}; grup konflik konten: {url_duplicate_summary.item(0, 'grup_konflik_konten'):,}.")
print("Kandidat IQR pada data sebelum deduplikasi:")
print(outlier_report)
print(f"Validasi SQL lulus: title/text/URL layak, URL unik, baris baca-ulang sesuai, checksum mentah tidak berubah = {raw_unchanged}.")


Baris awal: 1,149,789; baris akhir: 1,029,485.
Baris dikecualikan karena kelayakan: 0; baris dihapus karena URL berulang: 120,304.
Null asli per kolom: {'null_date': 0, 'null_link': 0, 'null_title': 0, 'null_text': 0, 'kosong_date': 0, 'kosong_link': 0, 'kosong_title': 0, 'kosong_text': 0}
Tanggal gagal parsing: 101,421 sebelum; 101,403 setelah deduplikasi URL; contoh: ['NaT'].
Duplikasi identik berlebih: 108,391; URL berlebih: 120,304; grup konflik konten: 86.
Kandidat IQR pada data sebelum deduplikasi:
shape: (2, 8)
┌──────────────┬────────┬────────┬────────┬─────────────┬────────────┬──────────────┬──────────────┐
│ fitur        ┆ Q1     ┆ Q3     ┆ IQR    ┆ batas_bawah ┆ batas_atas ┆ jumlah_kandi ┆ persen_kandi │
│ ---          ┆ ---    ┆ ---    ┆ ---    ┆ ---         ┆ ---        ┆ dat          ┆ dat          │
│ str          ┆ f64    ┆ f64    ┆ f64    ┆ f64         ┆ f64        ┆ ---          ┆ ---          │
│              ┆        ┆        ┆        ┆             ┆            ┆ i